In [ ]:
# BEMERKUNG – End-to-End Inference Pipeline (optional, nur zum eigenen Testen und Ausprobieren)

In [ ]:
# Komplette Inference Pipeline für Unwetterwarnung
import os
from datetime import datetime, timedelta
from pathlib import Path

import hopsworks
import joblib
import pandas as pd
from dotenv import load_dotenv


def load_project():
    project_root = Path.cwd().resolve()
    if not (project_root / ".env").exists():
        project_root = project_root.parent
    load_dotenv(project_root / ".env")

    api_key = os.getenv("HOPSWORKS_API_KEY")
    project_name = os.getenv("HOPSWORKS_PROJECT_NAME")
    if not api_key or not project_name:
        raise ValueError(
            "HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen in der .env-Datei gesetzt sein."
        )

    return hopsworks.login(
        api_key_value=api_key,
        project=project_name,
        host="eu-west.cloud.hopsworks.ai",
        port=443,
    )


def run_batch_prediction(model, batch_data: pd.DataFrame, threshold: float = 0.3) -> pd.DataFrame:
    expected_features = getattr(model, "feature_names_in_", None)
    if expected_features is None:
        expected_features = model.get_booster().feature_names
    if expected_features is None or len(expected_features) == 0:
        raise ValueError("Das Modell enthält keine Feature-Namen für die Inference.")

    missing_features = [
        feature_name for feature_name in expected_features
        if feature_name not in batch_data.columns
    ]
    if missing_features:
        raise KeyError(f"Fehlende Modell-Features: {missing_features}")

    model_input = batch_data.loc[:, list(expected_features)].apply(
        pd.to_numeric, errors="coerce"
    )
    probabilities = model.predict_proba(model_input)[:, 1]

    predictions_df = batch_data.copy()
    predictions_df["storm_probability"] = probabilities
    predictions_df["storm_warning"] = probabilities >= threshold
    predictions_df["risk_level"] = pd.Series(probabilities, index=predictions_df.index).map(
        lambda probability: "HOCH" if probability >= 0.6
        else "MITTEL" if probability >= threshold
        else "NIEDRIG"
    )
    return predictions_df


def run_inference_pipeline(project, locations: dict, threshold: float = 0.3):
    """
    End-to-End Inference Pipeline für Unwetterwarnung.
    """
    feature_store = project.get_feature_store()
    model_registry = project.get_model_registry()
    feature_view = feature_store.get_feature_view(
        name="severe_weather_fv", version=1
    )

    start_time = datetime.utcnow() - timedelta(hours=6)
    end_time = datetime.utcnow() + timedelta(days=3)
    batch_data = feature_view.get_batch_data(
        start_time=start_time, end_time=end_time
    ).fillna(0)

    model_meta = model_registry.get_model(
        name="severe_weather_classifier", version=1
    )
    model_dir = model_meta.download()
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    predictions_df = run_batch_prediction(model, batch_data, threshold=threshold)

    active_warnings = predictions_df[predictions_df["storm_warning"]]
    if not active_warnings.empty:
        print(f"🚨 {len(active_warnings)} Unwetterwarnungen aktiv")
        for _, row in active_warnings.iterrows():
            location = row.get("location", "Unbekannt")
            event_time = row.get("event_time", row.get("time", "Unbekannt"))
            print(
                f"  📍 {location} | 🕐 {event_time} | "
                f"⚡ Wahrscheinlichkeit: {row['storm_probability']:.1%}"
            )
    else:
        print("✅ Keine Unwetterwarnungen im Forecast-Zeitraum")

    return predictions_df


LOCATIONS = {
    "Munich": (48.1351, 11.5820),
    "Hamburg": (53.5511, 9.9937),
}
project = load_project()
predictions = run_inference_pipeline(project, LOCATIONS, threshold=0.3)

2026-09-14 14:43:57,523 INFO: Closing external client and cleaning up certificates.
2026-09-14 14:43:57,524 INFO: Connection closed.
2026-09-14 14:43:57,525 INFO: Initializing external client
2026-09-14 14:43:57,526 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-09-14 14:43:58,685 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/44167


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.06s) 
Using cached model files at '/tmp/hopsworks/models/fhnw_p1_weather_forcasts/severe_weather_classifier/1/severe_weather_classifier_1'. Pass local_path or call Model.clear_cache(...) to force a fresh download.
✅ Keine Unwetterwarnungen im Forecast-Zeitraum
